In [6]:
!git clone https://github.com/giuliomattolin/ConfMix.git

Cloning into 'ConfMix'...
remote: Enumerating objects: 141, done.
remote: Counting objects: 100% (141/141), done.
remote: Compressing objects: 100% (107/107), done.
remote: Total 141 (delta 42), reused 127 (delta 33), pack-reused 0 (from 0)
Receiving objects: 100% (141/141), 2.45 MiB | 8.04 MiB/s, done.
Resolving deltas: 100% (42/42), done.


In [7]:
%cd ConfMix

/kaggle/working/ConfMix


In [10]:
import yaml
import os

os.makedirs('data', exist_ok=True)

custom_data = {
    # دیتای Source (بازی) -> مدلی اولیه روی این آموزش می‌بیند
    'train': '/kaggle/input/studentdataset/student_dataset/train/images',  
    
    # دیتای Target Validation (واقعی) -> برای تست و ارزیابی دقت مدل
    'val': '/kaggle/input/studentdataset/student_dataset/test/images',      
    
    # دیتای Target UDA (واقعی) -> در فاز دوم (ConfMix) برای تطبیق دامنه استفاده می‌شود
    'uda': '/kaggle/input/studentdataset/student_dataset/aid_train/images',    
    
    'nc': 3,                                               
    'names': ['w', 'h0', 'h1']                                       
}

yaml_path = 'data/student_dataset.yaml'
with open(yaml_path, 'w') as f:
    yaml.dump(custom_data, f, sort_keys=False)

print("✅ فایل تنظیمات با مسیرهای اصلاح‌شده ساخته شد!")

✅ فایل تنظیمات با مسیرهای اصلاح‌شده ساخته شد!


In [11]:
import os

# مسیر پوشه پروژه
project_dir = '/kaggle/working/ConfMix'

# کلماتی که باید در کدهای YOLOv5 جایگزین شوند
replacements = {
    "torch.load(weights, map_location='cpu')": "torch.load(weights, map_location='cpu', weights_only=False)",
    "torch.load(w, map_location=map_location)": "torch.load(w, map_location=map_location, weights_only=False)",
    "torch.load(weights, map_location=device)": "torch.load(weights, map_location=device, weights_only=False)",
    "torch.load(f, map_location=map_location)": "torch.load(f, map_location=map_location, weights_only=False)"
}

# جستجو و اصلاح فایل‌های پایتون
for dname, dirs, files in os.walk(project_dir):
    for fname in files:
        if fname.endswith('.py'):
            fpath = os.path.join(dname, fname)
            with open(fpath, 'r', encoding='utf-8') as f:
                text = f.read()
            
            modified = False
            for old_str, new_str in replacements.items():
                if old_str in text:
                    text = text.replace(old_str, new_str)
                    modified = True
            
            if modified:
                with open(fpath, 'w', encoding='utf-8') as f:
                    f.write(text)
                print(f"✅ فایل اصلاح شد: {fname}")

print("🎉 تمام فایل‌ها با PyTorch 2.6 سازگار شدند!")

✅ فایل اصلاح شد: uda_train.py
✅ فایل اصلاح شد: train.py
🎉 تمام فایل‌ها با PyTorch 2.6 سازگار شدند!


In [12]:
import os
import re

project_dir = '/kaggle/working/ConfMix'

# جستجو و اصلاح فایل‌های پایتون
for dname, dirs, files in os.walk(project_dir):
    for fname in files:
        if fname.endswith('.py'):
            fpath = os.path.join(dname, fname)
            with open(fpath, 'r', encoding='utf-8') as f:
                text = f.read()
            
            # جایگزینی توابع منسوخ شده نامپای با عبارات استاندارد (با استفاده از Regex)
            new_text = re.sub(r'\bnp\.int\b', 'int', text)
            new_text = re.sub(r'\bnp\.float\b', 'float', new_text)
            new_text = re.sub(r'\bnp\.bool\b', 'bool', new_text)
            new_text = re.sub(r'\bnp\.object\b', 'object', new_text)
            
            if new_text != text:
                with open(fpath, 'w', encoding='utf-8') as f:
                    f.write(new_text)
                print(f"✅ فایل اصلاح شد: {fname}")

print("🎉 تمام فایل‌ها با نسخه جدید Numpy سازگار شدند!")

✅ فایل اصلاح شد: common.py
✅ فایل اصلاح شد: general.py
✅ فایل اصلاح شد: dataloaders.py
🎉 تمام فایل‌ها با نسخه جدید Numpy سازگار شدند!


In [16]:
!WANDB_MODE=disabled python train.py \
 --name game_pretrain \
 --batch 16 \
 --img 640 \
 --epochs 20 \
 --data data/student_dataset.yaml \
 --weights yolov5s.pt

2026-02-26 14:35:57.944745: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772116557.966997     328 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772116557.973981     328 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772116557.991477     328 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772116557.991505     328 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772116557.991509     328 computation_placer.cc:177] computation placer alr

In [17]:
import os

general_file = '/kaggle/working/ConfMix/utils/general.py'

with open(general_file, 'r', encoding='utf-8') as f:
    text = f.read()

# اصلاح تابع strip_optimizer
new_text = text.replace(
    "torch.load(f, map_location=torch.device('cpu'))", 
    "torch.load(f, map_location=torch.device('cpu'), weights_only=False)"
)

if new_text != text:
    with open(general_file, 'w', encoding='utf-8') as f:
        f.write(new_text)
    print("✅ فایل general.py برای همیشه اصلاح شد!")
else:
    print("⚠️ تغییری اعمال نشد.")

✅ فایل general.py برای همیشه اصلاح شد!


In [32]:
!WANDB_MODE=disabled python uda_train.py \
 --name game_adaptation \
 --batch 16 \
 --img 640 \
 --epochs 50 \
 --data data/student_dataset.yaml \
 --weights runs/train/game_pretrain5/weights/last.pt

2026-02-26 09:52:14.817967: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772099534.842038     959 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772099534.848875     959 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772099534.866291     959 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772099534.866321     959 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772099534.866327     959 computation_placer.cc:177] computation placer alr

In [19]:
import os

file_path = '/kaggle/working/ConfMix/models/experimental.py'

with open(file_path, 'r', encoding='utf-8') as f:
    content = f.read()

# اضافه کردن پارامتر weights_only=False به تابع torch.load
content = content.replace(
    "ckpt = torch.load(attempt_download(w), map_location=map_location)", 
    "ckpt = torch.load(attempt_download(w), map_location=map_location, weights_only=False)"
)

with open(file_path, 'w', encoding='utf-8') as f:
    f.write(content)
    
print("✅ فایل experimental.py با موفقیت پچ شد! مشکل امنیتی PyTorch برطرف شد.")

✅ فایل experimental.py با موفقیت پچ شد! مشکل امنیتی PyTorch برطرف شد.


In [ ]:
!python uda_val.py \
 --name game_evaluation \
 --img 640 \
 --data data/student_dataset.yaml \
 --weights runs/train/game_adaptation2/weights/last.pt \

uda_val: data=data/student_dataset.yaml, weights=['runs/train/game_adaptation2/weights/last.pt'], batch_size=32, imgsz=640, conf_thres=0.001, iou_thres=0.5, task=val, device=, workers=8, single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=False, project=runs/val, name=game_evaluation, exist_ok=False, half=False, dnn=False
YOLOv5 🚀 f4c0ef3 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)

Fusing layers... 
Model summary: 213 layers, 7029004 parameters, 0 gradients, 15.8 GFLOPs
val: Scanning '/kaggle/input/studentdataset/student_dataset/test/labels' images 
val: WARNING: Cache directory /kaggle/input/studentdataset/student_dataset/test is not writeable: [Errno 30] Read-only file system: '/kaggle/input/studentdataset/student_dataset/test/labels.cache.npy'
               Class     Images     Labels          P          R     mAP@.5 mAP@
                 all        495       1284      0.473      0.453      0.41

In [21]:
import yaml

# مسیر فایل تنظیمات (مسیر دقیق پوشه پروژه خود در کگل را جایگزین کنید)
# معمولاً چیزی شبیه به این است: '/kaggle/working/yolov5/data/hyps/hyp.scratch-low.yaml'
file_path = 'data/hyps/hyp.scratch-low.yaml' 

# خواندن تنظیمات فعلی
with open(file_path, 'r') as f:
    hyp = yaml.safe_load(f)

# اعمال تغییرات طلایی شما
hyp['lr0'] = 0.001
hyp['warmup_epochs'] = 5.0
hyp['mosaic'] = 0.5
hyp['mixup'] = 0.0
hyp['fl_gamma'] = 1.5  # فعال‌سازی Focal Loss

# ذخیره مجدد فایل با تنظیمات جدید
with open(file_path, 'w') as f:
    yaml.dump(hyp, f, default_flow_style=False)

print("✅ فایل YAML با موفقیت آپدیت شد و آماده آموزش است!")

✅ فایل YAML با موفقیت آپدیت شد و آماده آموزش است!


In [64]:
!python uda_val.py \
  --data data/student_dataset.yaml \
  --weights runs/train/game_adaptation_focal/weights/best.pt \
  --batch-size 32 \
  --imgsz 640 \
  --task val \
  --device 0 \
  --project runs/val \
  --name game_eval_focal

uda_val: data=data/student_dataset.yaml, weights=['runs/train/game_adaptation_focal/weights/best.pt'], batch_size=32, imgsz=640, conf_thres=0.001, iou_thres=0.6, task=val, device=0, workers=8, single_cls=False, augment=False, verbose=False, save_txt=False, save_hybrid=False, save_conf=False, save_json=False, project=runs/val, name=game_eval_focal, exist_ok=False, half=False, dnn=False
YOLOv5 🚀 f4c0ef3 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)

Fusing layers... 
YOLOv5s summary: 213 layers, 7029004 parameters, 0 gradients, 15.8 GFLOPs
val: Scanning '/kaggle/input/studentdataset/student_dataset/test/labels' images 
val: WARNING: Cache directory /kaggle/input/studentdataset/student_dataset/test is not writeable: [Errno 30] Read-only file system: '/kaggle/input/studentdataset/student_dataset/test/labels.cache.npy'
               Class     Images     Labels          P          R     mAP@.5 mAP@
                 all        495       1284      0.617      0.552  

In [63]:
!python uda_train.py \
  --data data/student_dataset.yaml \
  --cfg models/yolov5s.yaml \
  --weights runs/train/game_pretrain2/weights/best.pt \
  --hyp data/hyps/hyp.scratch-low.yaml \
  --epochs 40 \
  --patience 15 \
  --batch-size 32 \
  --imgsz 640 \
  --device 0 \
  --project runs/train \
  --name game_adaptation_focal

2026-02-26 13:43:43.296537: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1772113423.317705    7497 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1772113423.324245    7497 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1772113423.342140    7497 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772113423.342173    7497 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1772113423.342176    7497 computation_placer.cc:177] computation placer alr

In [65]:
!python detect.py \
  --weights runs/train/game_adaptation_focal/weights/best.pt \
  --source /kaggle/input/studentdataset/student_dataset/test/images \
  --conf 0.25 \
  --imgsz 640 \
  --device 0 \
  --project runs/detect \
  --name game_inference_focal \
  --save-txt

detect: weights=['runs/train/game_adaptation_focal/weights/best.pt'], source=/kaggle/input/studentdataset/student_dataset/test/images, data=data/coco128.yaml, imgsz=[640, 640], conf_thres=0.25, iou_thres=0.45, max_det=1000, device=0, view_img=False, save_txt=True, save_conf=False, save_crop=False, nosave=False, classes=None, agnostic_nms=False, augment=False, visualize=False, update=False, project=runs/detect, name=game_inference_focal, exist_ok=False, line_thickness=3, hide_labels=False, hide_conf=False, half=False, dnn=False
YOLOv5 🚀 f4c0ef3 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)

Fusing layers... 
YOLOv5s summary: 213 layers, 7029004 parameters, 0 gradients, 15.8 GFLOPs
image 1/495 /kaggle/input/studentdataset/student_dataset/test/images/00enhaced_19b3cf2ba42ce9d0935858d1125f8072.jpg: 480x640 Done. (0.034s)
image 2/495 /kaggle/input/studentdataset/student_dataset/test/images/00enhaced_54593362.jpg: 640x384 1 w, 1 h1, Done. (0.035s)
image 3/495 /kagg